# CUB-200 Aggregate Results

Attach the five per-model CUB output datasets, then run this notebook to produce the generalization tables and figures.

In [ ]:
%pip install -q timm captum grad-cam scikit-learn scipy seaborn

from pathlib import Path
import shutil, subprocess, sys

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
PY = sys.executable

def input_roots():
    roots = [WORK]
    if INPUT.exists():
        roots += [p for p in INPUT.iterdir() if p.is_dir()]
        datasets = INPUT / "datasets"
        if datasets.exists():
            for owner in datasets.iterdir():
                if owner.is_dir():
                    roots += [p for p in owner.iterdir() if p.is_dir()]
    return roots

def looks_like_code(p):
    return (p / "cub_200_generalization" / "aggregate_cub_results.py").exists()

def find_code_source():
    candidates = [INPUT / "spinexnet-code", INPUT / "spinexnet-code" / "het-spine"]
    for r in input_roots():
        candidates += [r, r / "het-spine", r / "spinexnet-code"]
    for p in candidates:
        if looks_like_code(p):
            return p
    raise FileNotFoundError("Could not find spinexnet-code with cub_200_generalization/aggregate_cub_results.py")

SRC = find_code_source()
CODE = WORK / "spinexnet-code"
if SRC.resolve() != CODE.resolve():
    shutil.copytree(SRC, CODE, dirs_exist_ok=True)

cmd = [
    PY,
    CODE / "cub_200_generalization" / "aggregate_cub_results.py",
    "--copy-inputs",
    "--output-dir",
    WORK / "cub_aggregate",
]
print("$", " ".join(map(str, cmd)), flush=True)
subprocess.run(list(map(str, cmd)), check=True)
